In [1]:
import re
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

ENV = "PROD"
df = pd.read_csv("qc_applicants_PROD.csv")

print(f"{len(df):,} rows | {df['docdb_family_id'].nunique():,} families | {df['psn_id'].nunique():,} applicants (psn_id)")
df.head()

20,895 rows | 12,596 families | 6,056 applicants (psn_id)


,docdb_family_id,psn_id,psn_name,han_name,psn_sector,applicant_ctry,n_applicants,family_year,offices,n_offices,granted_any,family_size,fwd_citations,first_office
0,65919538,21112,▲復▼旦大学ＦＵＮＤＡＮ ＵＮＩＶＥＲＳＩＴＹ,▲復▼旦大学Ｆｕｎｄａｎ Ｕｎｉｖｅｒｓｉｔｙ,UNIVERSITY,NaN,5,2018,CN EP JP US WO,5,True,5,8,CN
1,76545393,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,1,2018,US,1,False,1,6,US
2,71102213,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,5,2018,CN EP JP US WO,5,True,5,10,WO
3,72515606,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,1,2018,US,1,False,1,5,US
4,66663731,41455,1QB INFORMATION TECHNOLOGIES,1QB INFORMATION TECHNOLOGIES INC.,COMPANY,NaN,4,2017,CA CN EP JP US WO,6,True,7,45,WO


In [2]:
LEGAL = ["INC", "LLC", "LLLP", "LL", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "COPMANY", "GMBH", "AG", "SE",
         "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "NA", "OY", "AB", "AS", "ASA", "PTY", "PLC", "KK", "LLP", "LP",
         "PTE", "KG", "OOO", "SP Z OO", "SPOLKA", "JOINT.STOCK", "ANONIM SIRKETI", "OBSHCHESTVO"]
LEGAL_CJK = ["有限公司", "株式会社", "股份", "公司", "会社", "주식회사", "インコーポレイテッド", "アクチボラグ"]
UNIV = ["UNIV", "COLLEGE", "POLYTECHN", "POLITECNIC", "ECOLE", "HOCHSCHULE", "SCHOOL", "INSTITUTE OF TECHNOLOGY", "GIJUKU",
        "TRUSTEES", "EDUCATIONAL FUND", "大学", "大學", "学院", "カレッジ", "義塾", r"\bETH\b", r"\bEPFL\b", r"\bUESTC\b"]
GOV = ["INSTIT", "INSTYTUT", "INSTUTUC", "ИНСТИТУТ", "ACADEM", "COMMISSARIAT", "CNRS", "LABORATORY", "CENTER", "CENTRE",
       "CENTRUM", "ZENTRUM", "FOUNDATION", "FONDAZIONE", "FUNDACIO", "STIFTUNG", "FRAUNHOFER", "FROUNHAV", "PLANCK",
       "GOVERNMENT", "MINISTRY", "MINISTERIUM", "BUNDES", "FEDERATSIYA", "AGENTSTVO", "UCHREZHDENIE", "SECRETARY", "AGENCY",
       "COUNCIL", "CONSEJO", "CONSIGLIO", r"\bNATIONAL\b", "IDRYMA", "ASSOCIATION", "HOSPITAL", "ARMY", "AIR FORCE", "NAVY",
       "RIKEN", r"\bTNO\b", "NASA", "研究", "中心", "委员会", "科学院", "センター"]
FIRM = ["TECHNOLOG", "QUANT", "COMPUT", "SYSTEM", "SOLUTION", r"\bLABS?\b", "SOFTWARE", "NETWORK", "ELECTRONIC", "SEMICONDUCT",
        "PHOTONIC", "DEVICES", "INDUSTR", "HOLDING", "GROUP", "CAPITAL", "VENTURES", "ENTERPRISE", "INTERNATIONAL", "SCIENCE",
        "DIGITAL", "DATA", "INTELLIGEN", "ENERGY", "PHARMA", "SECUR", "MACHINES", "INSTRUMENTS", "ENGINEERING", "INNOVATION",
        "BLOCKCHAIN", "BANK", "INSURANCE", "PORTFOLIO", r"\bAI\b", "&", r"\+", "科技", "技术", "量子"]

LEGAL_RE = re.compile(r"\b(" + "|".join(LEGAL) + r")\b|" + "|".join(LEGAL_CJK))
UNIV_RE, GOV_RE, FIRM_RE = (re.compile("|".join(words)) for words in (UNIV, GOV, FIRM))
WORD = r"[^\W\d_]+(?:['\-][^\W\d_]+)*"                              # letters only, any alphabet; O'NEIL, JEAN-PAUL
PERSON_COMMA = re.compile(rf"^{WORD}( {WORD}){{0,3}} ?, ?{WORD}")     # 'SURNAME, Name', 'VAN LAER, Raphael'
PERSON_WORDS = re.compile(rf"^{WORD}( {WORD}){{1,3}}$")               # 'LESCANNE Raphael', 'Taylor Lee Patti'


def normalise(name):
    "Upper case, no accents, full-width characters -> ASCII, no dots, no '?' (characters lost in PATSTAT)."
    s = unicodedata.normalize("NFKC", str(name))
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).upper()
    s = " ".join(s.replace("?", "").replace(".", "").split())
    return s.replace("UNIVERSITY CORPORATION", "UNIVERSITY")             # Japanese national universities


def type_from_name(name):
    n = normalise(name)
    if LEGAL_RE.search(n): return "company"
    if UNIV_RE.search(n):  return "university"
    if GOV_RE.search(n):   return "gov / non-profit"
    if FIRM_RE.search(n):  return "company"
    if PERSON_COMMA.match(n) or PERSON_WORDS.match(n) or "·" in n:   # '·' separates names in Chinese transliterations
        return "individual"
    if len(n.split()) == 1: return "company"
    return "uncertain"


def type_from_patstat(sector):
    s = str(sector).upper()
    if "UNIVERSITY" in s:             return "university"
    if "GOV" in s or "HOSPITAL" in s: return "gov / non-profit"
    if "COMPANY" in s:                return "company"
    if "INDIVIDUAL" in s:             return "individual"
    return None                                                        # UNKNOWN or empty


applicants = df.groupby("psn_id").agg(psn_name=("psn_name", "first"), psn_sector=("psn_sector", "first"),
                                      families=("docdb_family_id", "nunique"))
applicants["type"] = applicants["psn_sector"].map(type_from_patstat)
from_name = applicants["type"].isna()
applicants.loc[from_name, "type"] = applicants.loc[from_name, "psn_name"].map(type_from_name)
df["type"] = df["psn_id"].map(applicants["type"])

# ---- families: each family counted once ----

step0_ids = pd.read_csv(Path.home() / "Group15_innovation/outputs/step0" / f"qc_families_{ENV}.csv")["docdb_family_id"].unique()
types_of_family = df.groupby("docdb_family_id")["type"].agg(lambda s: sorted(set(s)))
n_types = types_of_family.str.len()
print(f"families in Step 0:                       {len(step0_ids):6,}")
print(f"  without any applicant in PATSTAT:       {len(step0_ids) - len(types_of_family):6,}   (dropped)")
print(f"  with at least one applicant:            {len(types_of_family):6,}")
for t in ["company", "university", "gov / non-profit", "individual", "uncertain"]:
    print(f"    only {t:<34}{(types_of_family.str.join('') == t).sum():6,}")
print(f"    2 or more types                       {(n_types > 1).sum():6,}")

# ---- applicants per type; a family is counted in every type it has ----
summary = pd.DataFrame({
    "applicants": applicants["type"].value_counts(),
    "of which from the name rule": applicants.loc[from_name, "type"].value_counts(),
    "families with at least one": df.groupby("type")["docdb_family_id"].nunique(),
}).fillna(0).astype(int).reindex(["company", "university", "gov / non-profit", "individual", "uncertain"])
display(summary)

uncertain = applicants[applicants["type"] == "uncertain"].sort_values("families", ascending=False)
print(f"{len(uncertain)} uncertain applicants:")
uncertain[["psn_name", "families"]]

families in Step 0:                       12,630
  without any applicant in PATSTAT:           34   (dropped)
  with at least one applicant:            12,596
    only company                            7,940
    only university                         2,337
    only gov / non-profit                     559
    only individual                           415
    only uncertain                              0
    2 or more types                        1,345


,applicants,of which from the name rule,families with at least one
type,,,
company,2598,1604,8977
university,855,243,2989
gov / non-profit,312,157,880
individual,2287,895,1181
uncertain,4,4,15


4 uncertain applicants:


,psn_name,families
psn_id,,
23866221,-NOT AVAILABLE-,12
6733642,DHITECH - DISTRETTO TECNOLOGICO H IGH TECH S.C...,1
7821231,EHNSEHRM (EHNSTITJU NAS'ONAL' DE LJA SANTE EH ...,1
187402713,PEARL COHEN ZEDEK LATZER BARATZ,1


In [3]:
applicants.loc[[6733642, 187402713], "type"] = "company"   # DHITECH (consortium), PEARL COHEN (law firm)
applicants.loc[7821231, "type"] = "gov / non-profit"       # INSERM, transliterated
df["type"] = df["psn_id"].map(applicants["type"])

display(applicants[from_name & (applicants["type"] == "individual")].nlargest(10, "families")[["psn_name", "families"]])

org = df[(df["type"] != "individual") & (df["psn_name"] != "-NOT AVAILABLE-")].copy()
print(f"families:   {df['docdb_family_id'].nunique():,} -> {org['docdb_family_id'].nunique():,} with at least one organisation")
print(f"applicants: {df['psn_id'].nunique():,} -> {org['psn_id'].nunique():,} organisations (psn_id)")

,psn_name,families
psn_id,,
195262582,LESCANNE Rapha?l,7
192915572,Taylor Lee Patti,5
193942547,Elad Mentovich,5
194074734,Yonatan Piasetzky,5
194100481,Tali Septon,5
27646824,ROSE GEORDIE,4
186101699,"Dai, Haitao O.",4
189663297,Dr. Vishal Sharma,4
193314689,Eliahu Cohen,4


families:   12,596 -> 12,181 with at least one organisation
applicants: 6,056 -> 3,768 organisations (psn_id)


In [4]:
df.head()

,docdb_family_id,psn_id,psn_name,han_name,psn_sector,applicant_ctry,n_applicants,family_year,offices,n_offices,granted_any,family_size,fwd_citations,first_office,type
0,65919538,21112,▲復▼旦大学ＦＵＮＤＡＮ ＵＮＩＶＥＲＳＩＴＹ,▲復▼旦大学Ｆｕｎｄａｎ Ｕｎｉｖｅｒｓｉｔｙ,UNIVERSITY,NaN,5,2018,CN EP JP US WO,5,True,5,8,CN,university
1,76545393,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,1,2018,US,1,False,1,6,US,company
2,71102213,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,5,2018,CN EP JP US WO,5,True,5,10,WO,company
3,72515606,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,1,2018,US,1,False,1,5,US,company
4,66663731,41455,1QB INFORMATION TECHNOLOGIES,1QB INFORMATION TECHNOLOGIES INC.,COMPANY,NaN,4,2017,CA CN EP JP US WO,6,True,7,45,WO,company


In [20]:

df_firms = df[
    (df["type"].str.lower() != "individual") &
    (df["psn_name"] != "-NOT AVAILABLE-")
].copy()

df_firms["comp"] = (
    df_firms["han_name"]
    .replace(["", "nan", "None", "NULL"], np.nan)
    .fillna(df_firms["psn_name"])
    .astype(str)
    .str.strip()
)

df_partenza = (
    df_firms[["comp", "type"]]
    .drop_duplicates(subset=["comp"])
    .sort_values("comp")
    .reset_index(drop=True)
)

df_partenza.to_csv("organisations_applicants.csv", index=False, encoding="utf-8-sig")

print(f"Righe nel df completo (organizzazioni): {len(df_firms):,}")
print(f"Applicant unici salvati nel CSV di partenza: {len(df_partenza):,}")
print(df_partenza.head(10))

Righe nel df completo (organizzazioni): 17,751
Applicant unici salvati nel CSV di partenza: 3,110
                                                comp     type
0                       01 COMMUNIQUE LABORATORY INC  company
1                                    1372934 B C LTD  company
2                                  1372934 B.C. LTD.  company
3                                        1AHEAD TECH  company
4                           1QB INFORMATION TECH INC  company
5                   1QB INFORMATION TECHNOLOGIES INC  company
6                  1QB INFORMATION TECHNOLOGIES INC.  company
7                 1QB INFORMATION TECHNOLOGY COMPANY  company
8                                          1QB信息技术公司  company
9  7TH-O-5 RESEARCH INSTITUTE OF CHINESE SHIP HEA...  company


funzione clean - Traduzione dei maggiori  ascii del quantum

In [46]:
FORMS = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PLC", "GMBH", "MBH",
         "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE",
         "KK", "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA"]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"

WORDS = {"UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV", "UNIVERSIDAD": "UNIV",
         "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST", "INSTITUTO": "INST", "TECHNOLOGY": "TECH",
         "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH", "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB",
         "CENTRE": "CENTER", "SCIENCES": "SCIENCE", "大學": "大学"}

FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}

CJK_TO_ASCII = {
    "国际商业机器": "IBM",
    "谷歌": "GOOGLE",
    "微软技术许可": "MICROSOFT TECH LICENSING",
    "爱奥尼克": "IONQ",
    "耶鲁大学": "YALE UNIV",
    "霍尼韦尔国际": "HONEYWELL INTERNATIONAL",
    "特拉量子股份": "TERRA QUANTUM",
    "杜克大学": "DUKE UNIV",
    "英特尔": "INTEL",
    "康宁": "CORNING",
}
# 1. Termini istituzionali, accademici e di governance (standard OECD HAN / PATSTAT)
STOP_INSTITUTIONAL = {
    "UNIV", "INST", "COLLEGE", "SCHOOL", "ACADEMY", "POLITECNICO", "ECOLE", "HOCHSCHULE",
    "CENTER", "LABORATORY", "LAB", "RESEARCH", "FOUNDATION", "ASSOCIATION", "COUNCIL",
    "MINISTRY", "AGENCY", "HOSPITAL", "ARMY", "NAVY", "AIR",
    "TRUSTEES", "PRESIDENT", "BOARD"
}

# 2. Aggettivi generici, direzionali e descrittori di settore ad alta frequenza
STOP_DESCRIPTORS = {
    "NATIONAL", "INTERNATIONAL", "GENERAL", "AMERICAN", "UNITED", "ROYAL", "STATE", "CENTRAL",
    "NEW", "NORTH", "SOUTH", "EAST", "WEST", "GLOBAL", "FIRST", "ADVANCED", "APPLIED",
    "TECH", "QUANTUM", "DIGITAL", "SMART", "INFORMATION", "ELECTRONIC", "AEROSPACE",
    "BIO", "MED", "PHARMA", "ENERGY", "SOLAR", "POWER", "MICRO", "NANO", "OPTO", "ELECTRO", "AUTO", "AERO", "CHEM"
}

# 3. Toponimi nazionali e divisioni amministrative (es. Province/Municipalità cinesi ISO 3166-2:CN e capoluoghi)
STOP_GEO = {
    "CHINA", "CHINESE", "JAPAN", "NIPPON", "KOREA",
    # Municipalità e Province della Cina (standard amministrativo)
    "BEIJING", "TIANJIN", "SHANGHAI", "CHONGQING",
    "ANHUI", "FUJIAN", "GANSU", "GUANGDONG", "GUIZHOU", "HAINAN", "HEBEI", "HEILONGJIANG",
    "HENAN", "HUBEI", "HUNAN", "JIANGSU", "JIANGXI", "JILIN", "LIAONING", "QINGHAI",
    "SHAANXI", "XIAN", "SHANDONG", "SHANXI", "SICHUAN", "YUNNAN", "ZHEJIANG",
    "GUANGXI", "NEI", "MONGOL", "NINGXIA", "XINJIANG", "XIZANG", "TIBET",
    # Principali poli tecnologici / capoluoghi di provincia
    "SHENZHEN", "GUANGZHOU", "NANJING", "SUZHOU", "HANGZHOU", "NINGBO", "HEFEI",
    "WUHAN", "CHENGDU", "DALIAN", "QINGDAO", "CHANGCHUN", "HARBIN", "SHENYANG",
    "ZHENGZHOU", "CHANGSHA", "FUZHOU", "XIAMEN", "JINAN", "KUNMING", "TAIYUAN"
}

STOP_FIRST_WORDS = STOP_INSTITUTIONAL | STOP_DESCRIPTORS | STOP_GEO

def clean(name):
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "\u0250")))
    s = re.sub(r"[▲▼?.'’•·]", "", s)
    
    # Se c'è parte CJK seguita da parte Latina, tiene la parte Latina
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)
    # Se inizia con un prefisso/sigla in caratteri latini (es. "1QB信息技术", "ID量子技术", "IQM芬兰") seguito da ideogrammi, estrae la parte latina
    latin_prefix = re.match(r"^([A-Z0-9][A-Z0-9\- ]{1,15})[^\x00-\x7f]+$", s)
    if latin_prefix and len(re.sub(r"[^A-Z0-9]", "", latin_prefix.group(1))) >= 2:
        s = latin_prefix.group(1).strip()
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
        
    s = re.sub(r"\bC/O\b.*$", "", s).replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip()).strip()
    
    # Traduzione dizionario CJK -> ASCII
    s_compact = s.replace(" ", "")
    if s_compact in CJK_TO_ASCII:
        s = CJK_TO_ASCII[s_compact]
        
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    return re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s

Pipeline di armonizzazione 

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import AgglomerativeClustering

def pipeline_harm(df_input, col_nome, output_csv=None, alpha_word=0.6, max_dist=0.50):
    df_u = df_input[[col_nome]].drop_duplicates().copy()
    df_u["clean"] = df_u[col_nome].map(clean)
    
    # first word
    fw = df_u["clean"].str.split().str[0].fillna("")
    fw_counts = fw.value_counts()
    is_valid_fw = (fw.str.len() >= 3) & (~fw.str.isdigit()) & (~fw.isin(STOP_FIRST_WORDS)) & (fw.map(fw_counts) >= 2)
    
    etichetta = pd.Series(index=df_u.index, dtype="object")
    etichetta[is_valid_fw] = "FW_" + fw[is_valid_fw]
    
    # TF-IDF
    unmerged = df_u.loc[~is_valid_fw, "clean"]
    unique_u = unmerged.unique()

    if len(unique_u) > 1:
        sim_w = cosine_similarity(TfidfVectorizer(analyzer="word", token_pattern=r"(?u)\b\w+\b").fit_transform(unique_u))
        sim_c = cosine_similarity(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4)).fit_transform(unique_u))
        dist = 1 - (alpha_word * sim_w + (1 - alpha_word) * sim_c)
        np.fill_diagonal(dist, 0)
        c_ids = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=max_dist
        ).fit_predict(dist)
        etichetta[~is_valid_fw] = unmerged.map({n: f"TFIDF_{cid}" for n, cid in zip(unique_u, c_ids)})
    elif len(unique_u) == 1:
        etichetta[~is_valid_fw] = "TFIDF_0"
        
    # ID numerico (1, 2, 3...) 
    df_u["group_id"] = pd.factorize(etichetta)[0] + 1
    df_u["_is_fw"] = is_valid_fw
    df_u["_fw"] = fw
    
    def scegli_nome(sub):
        if sub["_is_fw"].iloc[0]:
            return sub["_fw"].iloc[0]
        vc = sub["clean"].value_counts()
        top = vc[vc == vc.iloc[0]].index.tolist()
        return min(top, key=len)
        
    mappa_nomi = df_u.groupby("group_id")[["clean", "_is_fw", "_fw"]].apply(scegli_nome)
    df_u["nome_assegnato"] = df_u["group_id"].map(mappa_nomi)
    
    # Riaggancia al DataFrame originale per non perdere nessuna riga
    df_out = df_input.merge(df_u[[col_nome, "clean", "group_id", "nome_assegnato"]], on=col_nome, how="left")
    
    if output_csv:
        df_out.to_csv(output_csv, index=False, encoding="utf-8-sig")
        
    return df_out

In [48]:

# 1. Sul Gold Standard
gold = pd.read_csv("PATSTAT_500_label_datav4.csv")
gold_finale = pipeline_harm(gold, col_nome="comp", output_csv="gold_harm.csv")

# 2. Sul Quantum (organisations_applicants.csv)
qc_org = pd.read_csv("organisations_applicants.csv")
#firms
df_company = qc_org[qc_org["type"] == "company"].copy()
harm_company = pipeline_harm(df_company, col_nome="comp")
#uni
df_altro = qc_org[qc_org["type"] != "company"].copy()
harm_altro = pipeline_harm(df_altro, col_nome="comp")


qc_finale = pd.concat([harm_company, harm_altro], ignore_index=True)
qc_finale["group_id"] = pd.factorize(qc_finale["type"] + "_" + qc_finale["group_id"].astype(str))[0] + 1
qc_finale.to_csv("quantum_harm.csv", index=False, encoding="utf-8-sig")


validazione gold standard  - metriche prof 

In [49]:

from math import comb
def pairs_inside(group_sizes):
    "Number of pairs of names inside groups of the given sizes: n(n-1)/2 for each group."
    return sum(comb(int(n), 2) for n in group_sizes)

def precision_recall_f1(gold_groups, predicted_groups):
    "Pairwise precision, recall and F1 of a predicted grouping against the gold groups."
    df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(df.groupby(["gold", "pred"]).size())         # merged by you and in the gold standard
    tp_fp = pairs_inside(df.groupby("pred").size())                # all the pairs you merged
    tp_fn = pairs_inside(df.groupby("gold").size())                # all the pairs that belong together
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"groups found": df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}
print("Original groups:", gold["Group"].nunique())
gold = pd.read_csv("gold_harm.csv")
print(pd.Series(precision_recall_f1(gold["Group"], gold["group_id"])).round(3))

Original groups: 481
groups found    581.000
precision         0.965
recall            0.922
F1                0.943
dtype: float64


Validazione settore quantum 

In [63]:
# 1. Legge il CSV completo
df = pd.read_csv("quantum_harm.csv")

# 2. Estrae un campione casuale di 300 righe con SOLO le colonne originali ('comp' e 'type')
campione_claude = df[["comp", "type"]].sample(n=300, random_state=42).reset_index(drop=True)

# 3. Salva il CSV pronto da caricare su Claude
campione_claude.to_csv("sample_for_claude_300.csv", index=False, encoding="utf-8-sig")

In [65]:
import pandas as pd
from math import comb

# 1. Carica il file di Claude e il tuo file armonizzato completo
claude_df = pd.read_csv("sample_for_claude_300_gold.csv")      # metti il nome esatto del file scaricato da Claude
tuo_df = pd.read_csv("quantum_harm.csv")

# 2. Riaggancia i tuoi gruppi usando la colonna 'comp'
confronto = claude_df.merge(
    tuo_df[["comp", "group_id", "nome_assegnato"]],
    on="comp",
    how="left"
)

# 3. Calcola Precision, Recall e F1 sulle 300 righe
def pairs_inside(group_sizes):
    return sum(comb(int(n), 2) for n in group_sizes)

def precision_recall_f1(gold_groups, predicted_groups):
    df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(df.groupby(["gold", "pred"]).size())
    tp_fp = pairs_inside(df.groupby("pred").size())
    tp_fn = pairs_inside(df.groupby("gold").size())
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn if tp_fn else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"groups found": df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}

print(pd.Series(precision_recall_f1(confronto["gold_group_id"], confronto["group_id"])).round(3))

# 4. (Bonus) Mostra esattamente dove tu e Claude siete in disaccordo!
# Falsi positivi: tu li hai messi insieme, Claude li ha separati
fp_groups = confronto.groupby("group_id")["gold_group_id"].nunique()
print("\n--- FALSI POSITIVI (uniti da te, separati da Claude) ---")
print(confronto[confronto["group_id"].isin(fp_groups[fp_groups > 1].index)].sort_values("group_id")[["comp", "nome_assegnato", "gold_name"]])

# Falsi negativi: Claude li ha messi insieme, tu li hai separati
fn_groups = confronto.groupby("gold_group_id")["group_id"].nunique()
print("\n--- FALSI NEGATIVI (uniti da Claude, separati da te) ---")
print(confronto[confronto["gold_group_id"].isin(fn_groups[fn_groups > 1].index)].sort_values("gold_group_id")[["comp", "gold_name", "nome_assegnato"]])

groups found    278.000
precision         0.458
recall            0.500
F1                0.478
dtype: float64

--- FALSI POSITIVI (uniti da te, separati da Claude) ---
                                                  comp  \
114                                   CELESTIAL AI INC   
283                                             AI INC   
224        STATE GRID JIANGSU ELECTRIC POWER CO., LTD.   
152                     BEIJING ELECTRIC POWER COMPANY   
44   NO.15 INSTITUTE OF CHINA ELECTRONICS TECHNOLOG...   
5    CHINA ELECTRONICS TECHNOLOGY GROUP CORPORATION...   
261  STATE GRID ZHEJIANG ELECTRIC POWER CO., LTD. N...   
130  STATE GRID JIANGSU ELECTRIC POWER CO., LTD. NA...   
60   ECONOMIC TECHNOLOGY RESEARCH INSTITUTE OF STAT...   
56   ECONOMIC TECHNOLOGY RESEARCH INSTITUTE OF STAT...   
103  ECONOMIC AND TECHNICAL RESEARCH INSTITUTE OF S...   
291  INFORMATION & TELECOMMUNICATION BRANCH OF STAT...   
278  INFORMATION COMMUNICATION COMPANY OF STATE GRI...   
201                

Prendo i noi che hanno aggregato più di 5 nomi inisme (73 righe) e a mano dico se sono tutti lo stesso o no -> precision

In [56]:
import pandas as pd

SOGLIA_MIN = 2
N_CAMPIONE = 50
SEED = 42

df = pd.read_csv('quantum_harm.csv')


validazione = (
    df.groupby(['group_id', 'nome_assegnato'], as_index=False)
    .agg(
        n_comp_unici=('comp', 'nunique'),
        n_clean_unici=('clean', 'nunique'),
        n_righe_totali=('comp', 'count'),
        lista_nomi_sporchi=('comp', lambda s: ' | '.join(sorted(s.dropna().unique())))
    )
)


candidati = validazione[validazione['n_comp_unici'] >= SOGLIA_MIN]

validazione_campione = (
    candidati.sample(n=min(N_CAMPIONE, len(candidati)), random_state=SEED)
    .sort_values(by=['n_comp_unici', 'group_id'], ascending=[False, True])
    .reset_index(drop=True)
)


validazione_campione['is_same_entity'] = ''

output_path = 'validazione_precision_cluster.csv'
validazione_campione.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"Candidati totali (>= {SOGLIA_MIN} forme sporche): {len(candidati)}")
print(f"Estratti {len(validazione_campione)} cluster casuali in '{output_path}'.")

Candidati totali (>= 2 forme sporche): 594
Estratti 50 cluster casuali in 'validazione_precision_cluster.csv'.


In [59]:
val_compilato = pd.read_csv('precision_cluster_validato.csv')

# Precision semplice (cluster 100% puri / totale cluster validati)
precision_cluster = (val_compilato['is_same_entity'] == 1).mean()

print(f"Cluster Precision: {precision_cluster:.2%}")


Cluster Precision (non pesata): 62.00%


uso tf idf sul nome originale pe prendere nomi simili , faccio 50 coppie e vedo se ovevano andare insieme e io non li ho messi 

In [61]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42
N_CAMPIONE = 50

df = pd.read_csv('quantum_harm.csv').reset_index(drop=True)

# 1. Similarità con TF-IDF su n-grammi di caratteri (3-4)
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), lowercase=True)
X = vec.fit_transform(df['comp'].astype(str))
sim = (X @ X.T).toarray()

# 2. Filtra coppie con group_id diverso nel bucket 0.65 - 0.98
iu, ju = np.triu_indices(len(df), k=1)
mask = (df['group_id'].values[iu] != df['group_id'].values[ju]) & (sim[iu, ju] >= 0.65) & (sim[iu, ju] <= 0.98)
iu, ju, scores = iu[mask], ju[mask], sim[iu, ju][mask]

# 3. Campionamento casuale nel bucket con seed fisso
rng = np.random.default_rng(SEED)
idx = rng.choice(len(scores), size=min(N_CAMPIONE, len(scores)), replace=False)

out = (
    pd.DataFrame({
        'score': np.round(scores[idx], 4),
        'comp_1': df.loc[iu[idx], 'comp'].values,
        'comp_2': df.loc[ju[idx], 'comp'].values,
        'group_id_1': df.loc[iu[idx], 'group_id'].values,
        'group_id_2': df.loc[ju[idx], 'group_id'].values,
        'dovevano_stare_insieme': ''  # Compila: 1 = Sì, 0 = No
    })
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

out.to_csv('validazione_recall_tfidf.csv', index=False, encoding='utf-8-sig')
print(f"Candidati nel bucket: {len(scores)} | Estratte {len(out)} coppie casuali.")

Candidati nel bucket: 549 | Estratte 50 coppie casuali.


In [62]:
import pandas as pd

val = pd.read_csv("validazione_recall_tfidf_validato.csv")
fn = (val["dovevano_stare_insieme"] == 1).sum()
tot = val["dovevano_stare_insieme"].notna().sum()

print(f"Separati per sbaglio (andavano insieme): {fn} su {tot} ({fn / tot:.1%})")

Separati per sbaglio (andavano insieme): 22 su 50 (44.0%)
